# Supervised Learning - Stage 4: Binary Classification

**Target:** `urban_or_rural_area` (classify whether a collision occurred in an urban or rural area)

Binary classification unlocks ROC-AUC as an additional evaluation metric.

| Model | Notes |
|-------|-------|
| Logistic Regression | Natural fit for binary problems; outputs calibrated probabilities |
| Random Forest | Non-linear; robust to the small feature set |
| XGBoost | Uses `binary:logistic` objective; `scale_pos_weight` handles imbalance when SMOTE not applied |

---
# 1. Configuration and Imports

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn scipy pyyaml xgboost --quiet

In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')


def find_project_root(marker='config.yml'):
    current = Path().resolve()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f'Could not find {marker} in any parent directory')


ROOT_DIR = find_project_root()
NOTEBOOKS_DIR = ROOT_DIR / 'notebooks'

with open(ROOT_DIR / 'config.yml') as f:
    cfg = yaml.safe_load(f)

with open(NOTEBOOKS_DIR / 'notebook-config.yml') as f:
    nb_cfg = yaml.safe_load(f)

NB_CONFIG = {
    'figsize_wide':   nb_cfg['plotting']['figsize_wide'],
    'figsize_square': nb_cfg['plotting']['figsize_square'],
    'palette':        nb_cfg['plotting']['palette'],
}

sns.set_theme(style='whitegrid', palette=NB_CONFIG['palette'])
print(f'Project root: {ROOT_DIR}')
print('Configs loaded.')

---
# 2. Load Stage 2 Outputs

In [ ]:
with open(NOTEBOOKS_DIR / 'stage_outputs' / 'stage2-binary.pkl', 'rb') as f:
    stage2 = pickle.load(f)

X_train = stage2['X_train']
y_train = stage2['y_train']
X_test  = stage2['X_test']
y_test  = stage2['y_test']
smote_applied = stage2['smote_applied']

class_weight = None if smote_applied else 'balanced'
class_names  = [str(c) for c in sorted(y_test.unique())]

print(f'Training set:  {X_train.shape}')
print(f'Test set:      {X_test.shape}')
print(f'Classes:       {class_names}')
print(f'SMOTE applied: {smote_applied}')
print(f'class_weight:  {class_weight}')

---
# 3. Feature Scaling

Same rationale as Stage 3. Scaler fit on training set only.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'X_train_scaled: {X_train_scaled.shape}')
print(f'X_test_scaled:  {X_test_scaled.shape}')

---
# 4. Shared Utilities

In [ ]:
def evaluate(model_name, y_test, y_pred, class_names):
    """Compute and print standard classification metrics. Returns a result dict."""
    accuracy     = accuracy_score(y_test, y_pred)
    f1_weighted  = f1_score(y_test, y_pred, average='weighted')
    f1_macro     = f1_score(y_test, y_pred, average='macro')
    f1_per_class = f1_score(y_test, y_pred, average=None)

    print(f'--- {model_name} ---')
    print(f'Accuracy:    {accuracy:.4f}')
    print(f'Weighted F1: {f1_weighted:.4f}')
    print(f'Macro F1:    {f1_macro:.4f}')
    print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

    return {
        'model_name':   model_name,
        'accuracy':     accuracy,
        'f1_weighted':  f1_weighted,
        'f1_macro':     f1_macro,
        'f1_per_class': f1_per_class,
        'y_pred':       y_pred,
    }


def plot_confusion_matrix(ax, y_test, y_pred, class_names, title):
    """Plot a confusion matrix on the given axes."""
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=10, fontweight='bold')


cv = StratifiedKFold(
    n_splits=cfg['supervised']['cv_folds'],
    shuffle=True,
    random_state=cfg['supervised']['random_state'],
)

results = {}

---
# 5. Logistic Regression

In [ ]:
lr_cfg = cfg['supervised']['models']['logistic_regression']

lr_model = LogisticRegression(
    max_iter=lr_cfg['max_iter'],
    class_weight=class_weight,
    solver='lbfgs',
    random_state=cfg['supervised']['random_state'],
)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
results['Logistic Regression'] = evaluate('Logistic Regression', y_test, y_pred_lr, class_names)
results['Logistic Regression']['model']  = lr_model
results['Logistic Regression']['scaler'] = scaler

---
# 6. Random Forest

In [ ]:
rf_cfg = cfg['supervised']['models']['random_forest']

param_grid_rf = {
    'n_estimators':      rf_cfg['n_estimators'],
    'max_depth':         rf_cfg['max_depth'],
    'min_samples_split': rf_cfg['min_samples_split'],
}

base_rf = RandomForestClassifier(
    class_weight=class_weight,
    random_state=cfg['supervised']['random_state'],
    n_jobs=-1,
)

search_rf = GridSearchCV(
    estimator=base_rf,
    param_grid=param_grid_rf,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

print('Running GridSearchCV for Random Forest...')
search_rf.fit(X_train, y_train)

print(f'Best params:        {search_rf.best_params_}')
print(f'Best macro F1 (CV): {search_rf.best_score_:.4f}')

In [ ]:
rf_model  = search_rf.best_estimator_
y_pred_rf = rf_model.predict(X_test)
results['Random Forest'] = evaluate('Random Forest', y_test, y_pred_rf, class_names)
results['Random Forest']['model']       = rf_model
results['Random Forest']['best_params'] = search_rf.best_params_

In [ ]:
importances = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=tuple(NB_CONFIG['figsize_square']))
ax.barh(importances.index, importances.values,
        color=sns.color_palette(NB_CONFIG['palette'])[0])
ax.axvline(importances.mean(), color='red', linestyle='--', linewidth=1, label='Mean')
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('Random Forest - Top 15 Feature Importances', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
# 7. XGBoost

Binary task uses `binary:logistic` objective. When SMOTE was not applied,
`scale_pos_weight` is set to the negative/positive class ratio to compensate
for imbalance — this is XGBoost's equivalent of `class_weight='balanced'`.

In [ ]:
# Binary task - XGBoost uses binary:logistic objective
# scale_pos_weight compensates for imbalance when SMOTE was not applied
if not smote_applied:
    neg = (y_train == sorted(y_train.unique())[0]).sum()
    pos = (y_train == sorted(y_train.unique())[1]).sum()
    scale_pos_weight = neg / pos
    print(f'scale_pos_weight set to {scale_pos_weight:.2f} (neg/pos ratio)')
else:
    scale_pos_weight = 1
    print('scale_pos_weight=1 (SMOTE already balanced the training set)')

# Encode to 0/1
original_classes = sorted(y_train.unique())
label_map   = {orig: idx for idx, orig in enumerate(original_classes)}
inverse_map = {idx: orig for orig, idx in label_map.items()}

y_train_enc = y_train.map(label_map)

xgb_cfg = cfg['supervised']['models']['xgboost']

param_grid_xgb = {
    'learning_rate': xgb_cfg['learning_rate'],
    'max_depth':     xgb_cfg['max_depth'],
    'n_estimators':  xgb_cfg['n_estimators'],
    'subsample':     xgb_cfg['subsample'],
}

base_xgb = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=cfg['supervised']['random_state'],
    n_jobs=-1,
    verbosity=0,
)

search_xgb = GridSearchCV(
    estimator=base_xgb,
    param_grid=param_grid_xgb,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

print('Running GridSearchCV for XGBoost...')
search_xgb.fit(X_train, y_train_enc)

print(f'Best params:        {search_xgb.best_params_}')
print(f'Best macro F1 (CV): {search_xgb.best_score_:.4f}')

In [ ]:
xgb_model  = search_xgb.best_estimator_
y_pred_enc = xgb_model.predict(X_test)
y_pred_xgb = pd.Series(y_pred_enc).map(inverse_map).values

results['XGBoost'] = evaluate('XGBoost', y_test, y_pred_xgb, class_names)
results['XGBoost']['model']       = xgb_model
results['XGBoost']['best_params'] = search_xgb.best_params_
results['XGBoost']['label_map']   = label_map
results['XGBoost']['inverse_map'] = inverse_map

In [ ]:
importance_weight = pd.Series(
    xgb_model.get_booster().get_score(importance_type='weight'),
    name='weight'
).reindex(X_train.columns, fill_value=0).sort_values(ascending=True).tail(15)

importance_gain = pd.Series(
    xgb_model.get_booster().get_score(importance_type='gain'),
    name='gain'
).reindex(X_train.columns, fill_value=0).sort_values(ascending=True).tail(15)

fig, axes = plt.subplots(1, 2, figsize=tuple(NB_CONFIG['figsize_wide']))
palette = sns.color_palette(NB_CONFIG['palette'])

axes[0].barh(importance_weight.index, importance_weight.values, color=palette[0])
axes[0].axvline(importance_weight.mean(), color='red', linestyle='--', linewidth=1, label='Mean')
axes[0].set_xlabel('Weight (split count)')
axes[0].set_title('Top 15 by Weight', fontsize=11, fontweight='bold')
axes[0].legend()

axes[1].barh(importance_gain.index, importance_gain.values, color=palette[2])
axes[1].axvline(importance_gain.mean(), color='red', linestyle='--', linewidth=1, label='Mean')
axes[1].set_xlabel('Gain (avg loss improvement)')
axes[1].set_title('Top 15 by Gain', fontsize=11, fontweight='bold')
axes[1].legend()

fig.suptitle('XGBoost - Feature Importances', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 8. Model Comparison

In [ ]:
comparison_rows = []
for model_name, r in results.items():
    row = {
        'Model':       model_name,
        'Accuracy':    r['accuracy'],
        'Weighted F1': r['f1_weighted'],
        'Macro F1':    r['f1_macro'],
    }
    for cls_name, f1 in zip(class_names, r['f1_per_class']):
        row[f'F1 ({cls_name})'] = f1
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index('Model').round(4)
print('Model comparison:')
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_name, r) in zip(axes, results.items()):
    plot_confusion_matrix(ax, y_test, r['y_pred'], class_names, model_name)

fig.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
palette = sns.color_palette(NB_CONFIG['palette'], len(results))
bars = ax.bar(list(results.keys()),
              [r['f1_macro'] for r in results.values()],
              color=palette, edgecolor='white')
for bar, r in zip(bars, results.values()):
    ax.annotate(
        f'{r["f1_macro"]:.4f}',
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4), textcoords='offset points',
        ha='center', fontsize=10
    )
ax.set_ylabel('Macro F1')
ax.set_title('Macro F1 Comparison', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### ROC-AUC

Binary classification enables ROC-AUC — a threshold-independent measure of
discriminative ability that is more informative than accuracy under class imbalance.

In [ ]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.preprocessing import LabelBinarizer

# Binary task - compute ROC-AUC for each model that exposes predict_proba
fig, ax = plt.subplots(figsize=(7, 5))
palette = sns.color_palette(NB_CONFIG['palette'], len(results))

for (model_name, r), color in zip(results.items(), palette):
    model = r['model']
    X_eval = X_test_scaled if model_name == 'Logistic Regression' else X_test
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_eval)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        RocCurveDisplay.from_predictions(
            y_test, y_prob,
            name=f'{model_name} (AUC={auc:.3f})',
            ax=ax, color=color
        )

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_title('ROC Curves - Binary (urban_or_rural_area)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 9. Persist Stage 4 Outputs

In [ ]:
output_dir = NOTEBOOKS_DIR / 'stage_outputs'
os.makedirs(output_dir, exist_ok=True)

output_path = output_dir / 'stage4.pkl'
with open(output_path, 'wb') as f:
    pickle.dump({
        'task':          'binary',
        'target_col':    'urban_or_rural_area',
        'class_names':   class_names,
        'results':       {
            model_name: {
                'accuracy':     r['accuracy'],
                'f1_weighted':  r['f1_weighted'],
                'f1_macro':     r['f1_macro'],
                'f1_per_class': r['f1_per_class'],
                'y_pred':       r['y_pred'],
                'model':        r['model'],
                'best_params':  r.get('best_params', None),
            }
            for model_name, r in results.items()
        },
        'scaler':        scaler,
        'y_test':        y_test,
        'X_test':        X_test,
        'comparison_df': comparison_df,
    }, f)

print(f'Stage 4 outputs saved to: {output_path}')